In [ ]:
!pip install trl datasets peft bitsandbytes accelerate optuna evaluate

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [ ]:
args = SFTConfig()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/layer_skip/scripts')

In [ ]:
from custom_trainer import LayerSkipSFTTrainer  # or whatever class/function you're importing

In [ ]:
import inspect
from custom_trainer import LayerSkipSFTTrainer

print(inspect.getsource(LayerSkipSFTTrainer))

class LayerSkipSFTTrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.early_exit_layer = 0
        self.always_last_layer = True
        self.early_exit_loss_scale = 1.0

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        self.early_exit_layer = (
            self.early_exit_layer % (model.config.num_hidden_layers - 1)
        ) + 1

        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        labels = inputs.pop("labels")

        # Forward pass with hidden states
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        # Extract early-exit hidden state
        hidden_state = outputs["hidden_states"][self.early_exit_layer].to(model.dtype)

        # Try to apply final LayerNorm if available
        norm_layer = None
        try:
          

In [ ]:
# Model from Hugging Face hub
base_model = "meta-llama/Llama-3.2-1B"


# New instruction dataset

dataset = load_dataset("mlabonne/guanaco-llama2-1k")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Access the 'train' split
full_dataset = dataset["train"]

# Step 1: Shuffle the dataset (to avoid bias)
full_dataset = full_dataset.shuffle(seed=42)

# Step 2: Split into train, validation, and test
train_testvalid = full_dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = train_testvalid["train"]
test_valid_dataset = train_testvalid["test"]

# Split test_valid again into validation and test sets
test_valid_split = test_valid_dataset.train_test_split(test_size=0.5, seed=42)

eval_dataset = test_valid_split["train"]   # Validation set
test_dataset = test_valid_split["test"]     # Test set

# Final Sizes:
print(f"Train Size: {len(train_dataset)}")
print(f"Validation Size: {len(eval_dataset)}")
print(f"Test Size: {len(test_dataset)}")

Train Size: 800
Validation Size: 100
Test Size: 100


In [ ]:
from huggingface_hub import login

# Paste your HF token as a string
login(token="")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Tokenize
def tokenize(example):
    tokenized = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


tokenized_train = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized_val = eval_dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize, batched=True, remove_columns=["text"])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorWithPadding
from peft import get_peft_model, LoraConfig, TaskType

from transformers import DataCollatorForLanguageModeling




import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

def build_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_config,
        device_map="auto",
        token=True  # ✅ replace deprecated use_auth_token
    )

    # ✅ Required for LoRA + 4-bit
    model = prepare_model_for_kbit_training(model)
    # Optional manual norm patch if `prepare_model_for_kbit_training()` fails
    for name, module in model.named_modules():
        if "layernorm" in name.lower():
            module = module.to(torch.float32)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    model = get_peft_model(model, lora_config)
    model.train()

    return model





In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding, DataCollatorForLanguageModeling

model = build_model()

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 2. Define a formatting function
def formatting_prompts_func(example):
    return example["text"]

args = TrainingArguments(
    output_dir="./final-llm-training",
    per_device_train_batch_size=1,
    learning_rate=9.03742035898082e-05,
    weight_decay=0.18882404302177638,
    num_train_epochs=1,
    report_to="none",
    save_strategy="no",
    logging_strategy="steps",
    label_names=["labels"],
    logging_steps=100
)

# 6. Create Trainer
trainer =  LayerSkipSFTTrainer(
    model=model,
    train_dataset=tokenized_train,
    args=args
)


trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
100,4.473000
200,3.813800
300,3.614900
400,3.542800
500,3.247600
600,3.433000
700,3.383900
800,3.078200


TrainOutput(global_step=800, training_loss=3.5733935165405275, metrics={'train_runtime': 264.9633, 'train_samples_per_second': 3.019, 'train_steps_per_second': 3.019, 'total_flos': 2395791477964800.0, 'train_loss': 3.5733935165405275})

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create DataLoader
eval_loader = DataLoader(
    tokenized_test,
    batch_size=1,
    shuffle=False,
    collate_fn=data_collator
)


all_losses = []

with torch.no_grad():
    for batch in tqdm(eval_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        logits = outputs.logits

        # Assuming your model has a custom loss_function
        loss = model.loss_function(logits=logits, labels=labels, vocab_size=logits.size(-1))

        all_losses.append(loss.item())

# Compute mean loss and perplexity
eval_loss = sum(all_losses) / len(all_losses)
perplexity = torch.exp(torch.tensor(eval_loss))

print(f"Manual Eval Loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

100%|██████████| 100/100 [00:09<00:00, 10.02it/s]

Manual Eval Loss: 1.9758
Perplexity: 7.21


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding, DataCollatorForLanguageModeling

model = build_model()
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 2. Define a formatting function
def formatting_prompts_func(example):
    return example["text"]

args = TrainingArguments(
    output_dir="./final-llm-training",
    per_device_train_batch_size=1,
    learning_rate=9.03742035898082e-05,
    weight_decay=0.18882404302177638,
    num_train_epochs=1,
    report_to="none",
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=100
)

# 6. Create Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_train,
    args=args,
    data_collator=data_collator,
)


trainer.train()


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:476: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
100,1.945300
200,1.792200
300,1.832800
400,1.893000
500,1.781500
600,1.744700
700,1.749200
800,1.750300


TrainOutput(global_step=800, training_loss=1.8111294174194337, metrics={'train_runtime': 104.024, 'train_samples_per_second': 7.691, 'train_steps_per_second': 7.691, 'total_flos': 2393697681408000.0, 'train_loss': 1.8111294174194337, 'epoch': 1.0})

In [ ]:
model.push_to_hub('HassaanSeeker/llama-3.2-1b-guanco-finetuned-qlora-layerskip')
tokenizer.push_to_hub("HassaanSeeker/llama-3.2-1b-guanco-finetuned-layerskip")

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/6.83M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/HassaanSeeker/llama-3.2-1b-guanco-finetuned-qlora-layerskip/commit/775acf404b7bd1ee92a5f017543db4f5d64672af', commit_message='Upload tokenizer', commit_description='', oid='775acf404b7bd1ee92a5f017543db4f5d64672af', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HassaanSeeker/llama-3.2-1b-guanco-finetuned-qlora-layerskip', endpoint='https://huggingface.co', repo_type='model', repo_id='HassaanSeeker/llama-3.2-1b-guanco-finetuned-qlora-layerskip'), pr_revision=None, pr_num=None)

In [ ]:
import math

metrics = trainer.evaluate(eval_dataset=tokenized_test)
eval_loss = metrics["eval_loss"]

# eval_loss can sometimes be a tensor, ensure it's a float explicitly:
if isinstance(eval_loss, torch.Tensor):
    eval_loss = eval_loss.item()

perplexity = math.exp(eval_loss)
print(f"Eval loss: {eval_loss:.4f}, Perplexity: {perplexity:.2f}")

Eval loss: 1.5691, Perplexity: 4.80


In [ ]:
import evaluate

In [ ]:
perplexity = evaluate.load("perplexity", module_type="metric")

In [ ]:
texts = [t for t in list(dataset["validation"]["text"]) if t and t.strip()]  # List[str] of 3760 entries

# Compute perplexity using a pretrained model (e.g. gpt2)
results = perplexity.compute(
    model_id="HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned",  # You can also use your own path/to/finetuned-model
    predictions = texts,
    batch_size=1,           # Change based on your GPU RAM
    add_start_token=True    # Recommended for GPT-like models
)

config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
model.save_pretrained("Llama-3.2-1b-hf-layerskip-v2-finetuned")
tokenizer.save_pretrained("Llama-3.2-1b-hf-layerskip-v2-finetuned")

('Llama-3.2-1b-hf-layerskip-v2-finetuned/tokenizer_config.json',
 'Llama-3.2-1b-hf-layerskip-v2-finetuned/special_tokens_map.json',
 'Llama-3.2-1b-hf-layerskip-v2-finetuned/tokenizer.json')

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineG

In [ ]:
model.push_to_hub("HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned")

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned/commit/ae279d9cae765da3ae83ed1263a7e2016d3fddad', commit_message='Upload LlamaForCausalLM', commit_description='', oid='ae279d9cae765da3ae83ed1263a7e2016d3fddad', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned'), pr_revision=None, pr_num=None)

In [ ]:
tokenizer.push_to_hub("HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned")

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned/commit/f4cd72deadb0258fc1265cc1bc02c65e1f02dea6', commit_message='Upload tokenizer', commit_description='', oid='f4cd72deadb0258fc1265cc1bc02c65e1f02dea6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='HassaanSeeker/Llama-3.2-1b-hf-layerskip-v2-finetuned'), pr_revision=None, pr_num=None)

In [ ]:
torch.cuda.empty_cache()